In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
rng = np.random.default_rng(42)
N_SITES = 100

countries = ["Germany", "Spain", "France", "Italy", "Poland"]
indications = ["Oncology", "Cardiology", "Neurology", "Immunology"]

In [12]:
#site characteristics
site_data = []

for i in range(1, N_SITES + 1):
    site_id = f"SITE_{i:03d}"

    country = rng.choice(countries)

    # Each site has 2 to 4 distinct indications
    n_indications = rng.integers(2, len(indications) + 1)
    site_indications = rng.choice(
        indications,
        size=n_indications,
        replace=False
    )

    # Characteristics shared by all indications at this site
    activation_days = rng.integers(30, 181)
    staff_capacity = rng.integers(1, 11)
    data_quality_score = rng.integers(50, 101)

    # Generate one record per site-indication combination
    for indication in site_indications:
        site_data.append({
            "site_id": site_id,
            "country": country,
            "indication": indication,
            "eligible_patients_monthly": rng.integers(5, 41),
            "screen_failure_rate": round(rng.uniform(0.10, 0.40), 2),
            "competing_studies": rng.integers(0, 6),
            "activation_days": activation_days,
            "staff_capacity": staff_capacity,
            "data_quality_score": data_quality_score,
            "recruitment_rate": round(rng.uniform(0.10, 0.50), 2)
        })

df = pd.DataFrame(site_data)


In [13]:
#competition
df["competition_factor"] = (
    1 / (1 + 0.15 * df["competing_studies"])
).round(3)


In [14]:
#adjusted monthly recruitment potential
df["adjusted_recruitment"] = (
    df["eligible_patients_monthly"]
    * (1 - df["screen_failure_rate"])
    * df["recruitment_rate"]
    * df["competition_factor"]
).round(2)

In [15]:
#operational risk category
df["risk_level"] = np.select(
    [
        (df["activation_days"] > 120)
        | (df["data_quality_score"] < 65)
        | (df["staff_capacity"] <= 3),

        (df["activation_days"] > 90)
        | (df["data_quality_score"] < 80)
        | (df["staff_capacity"] <= 5)
    ],
    ["High", "Medium"],
    default="Low"
)


In [16]:
#validate dataset structure
assert df["site_id"].nunique() == N_SITES
assert df.groupby("site_id")["indication"].nunique().min() >= 2
assert not df.duplicated(["site_id", "indication"]).any()

In [ ]:
#output 
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "sites_data.csv"
df.to_csv(output_path, index=False)

In [19]:
#report results
indications_per_site = df.groupby("site_id")["indication"].nunique()

print(f"Generated {df['site_id'].nunique()} synthetic sites.")
print(f"Generated {len(df)} site-indication records.")
print(
    "Indications per site:",
    f"minimum={indications_per_site.min()},",
    f"maximum={indications_per_site.max()}"
)
print(f"Dataset saved to: {output_path}")
print("\nRecords per indication:")
print(df["indication"].value_counts().sort_index())
print("\nSample records:")
print(df.head(10))

Generated 100 synthetic sites.
Generated 298 site-indication records.
Indications per site: minimum=2, maximum=4
Dataset saved to: data\sites dataset.csv

Records per indication:
indication
Cardiology    76
Immunology    72
Neurology     76
Oncology      74
Name: count, dtype: int64

Sample records:
    site_id country  indication  eligible_patients_monthly  \
0  SITE_001   Spain    Oncology                         37   
1  SITE_001   Spain  Cardiology                         18   
2  SITE_002   Italy    Oncology                         21   
3  SITE_002   Italy  Cardiology                          5   
4  SITE_002   Italy   Neurology                         19   
5  SITE_002   Italy  Immunology                         35   
6  SITE_003   Italy  Cardiology                         31   
7  SITE_003   Italy  Immunology                         20   
8  SITE_003   Italy   Neurology                         12   
9  SITE_003   Italy    Oncology                         27   

   screen_failur